In [2]:
import pandas as pd
import numpy as np

## Removing Duplicates

Two methods handle duplicate rows in a DataFrame:

| Method | Returns |
|--------|--------|
| `duplicated()` | Boolean Series — `True` for rows that are duplicates of an earlier row |
| `drop_duplicates()` | DataFrame with duplicate rows removed |

### Default Behaviour

- All columns are compared by default.
- The **first** occurrence of each combination is kept; subsequent ones are marked as duplicates.

### Options

| Argument | Effect |
|----------|--------|
| `subset=[col, ...]` | Only consider these columns when detecting duplicates |
| `keep="first"` | Keep the first occurrence (default) |
| `keep="last"` | Keep the last occurrence instead |

## Key Points

- `duplicated` returns a boolean mask; `drop_duplicates` returns the filtered DataFrame.
- By default all columns are used for comparison; use `subset=` to narrow this.
- `keep="last"` retains the most recent occurrence of each duplicate combination.

In [3]:
data = pd.DataFrame({
    "k1": ["one", "two"] * 3 + ["two"],
    "k2": [1, 1, 2, 3, 3, 4, 4]
})

data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


In [4]:
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

In [5]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


In [6]:
data["v1"] = range(7)

data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [7]:
data.drop_duplicates(subset=["k1"])  # detect duplicates on k1 only

,k1,k2,v1
0,one,1,0
1,two,1,1


In [8]:
data.drop_duplicates(["k1", "k2"], keep="last")  # keep last occurrence

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


## Transforming Data Using a Function or Mapping

The `map` method on a Series applies an element-wise transformation using either:

- A **dictionary** — maps each existing value to a new value.
- A **function** — called on each element individually.

A common use case is deriving a new column from an existing one using a lookup dict.

## Key Points

- `series.map(dict)` replaces each value using the dictionary as a lookup table.
- `series.map(func)` calls `func` on every element.
- Both approaches are convenient for element-wise data cleaning and derivation.

In [9]:
data = pd.DataFrame({
    "food": ["bacon", "pulled pork", "bacon", "pastrami",
             "corned beef", "bacon", "pastrami", "honey ham", "nova lox"],
    "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]
})

data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


In [10]:
meat_to_animal = {
    "bacon":       "pig",
    "pulled pork": "pig",
    "pastrami":    "cow",
    "corned beef": "cow",
    "honey ham":   "pig",
    "nova lox":    "salmon"
}

data["animal"] = data["food"].map(meat_to_animal)

data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


In [11]:
# Equivalent using a function
def get_animal(x):
    return meat_to_animal[x]

data["food"].map(get_animal)

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: object

## Replacing Values

`replace` provides a flexible way to substitute specific values in a Series or DataFrame.

It is more general than `fillna` (which only targets `NaN`) — `replace` can target any value.

### Replace Patterns

| What you pass | Effect |
|---------------|--------|
| `replace(old, new)` | Replace one value with another |
| `replace([v1, v2], new)` | Replace multiple values with one replacement |
| `replace([v1, v2], [r1, r2])` | Replace each value with its own replacement |
| `replace({v1: r1, v2: r2})` | Same as above using a dictionary |

> **Note:** `data.replace` is distinct from `data.str.replace`, which does element-wise string substitution.

## Key Points

- `replace` targets any value, not just `NaN`.
- Pass a list to replace multiple values with one replacement.
- Pass parallel lists or a dict to map each old value to its own replacement.
- Returns a new object by default — does not modify in place.

In [12]:
data = pd.Series([1., -999., 2., -999., -1000., 3.])

data

0       1.0
1    -999.0
2       2.0
3    -999.0
4   -1000.0
5       3.0
dtype: float64

In [13]:
data.replace(-999, np.nan)  # single value replacement

0       1.0
1       NaN
2       2.0
3       NaN
4   -1000.0
5       3.0
dtype: float64

In [14]:
data.replace([-999, -1000], np.nan)  # multiple values → same replacement

0    1.0
1    NaN
2    2.0
3    NaN
4    NaN
5    3.0
dtype: float64

In [15]:
data.replace([-999, -1000], [np.nan, 0])  # each value gets its own replacement

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

In [16]:
data.replace({-999: np.nan, -1000: 0})  # dict form — equivalent to above

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

## Renaming Axis Indexes

Axis labels (index and column names) can be transformed similarly to values.

### In-Place via `index` Assignment

Apply a function to the index using its `map` method, then assign the result back:

```python
df.index = df.index.map(func)  # modifies the DataFrame in place
```

### Non-Modifying with `rename`

`rename` returns a new DataFrame with transformed labels — the original is unchanged.

- Pass a **function** to apply to all labels: `rename(index=str.title, columns=str.upper)`.
- Pass a **dict** to rename only a subset of labels: `rename(index={"OLD": "NEW"})`.

## Key Points

- `df.index.map(func)` produces a new Index; assigning it back modifies the DataFrame in place.
- `rename` is the preferred approach when you want a new object without touching the original.
- `rename` accepts functions (applied to all labels) or dicts (applied to a subset).
- Both `index=` and `columns=` can be passed to `rename` simultaneously.

In [17]:
data = pd.DataFrame(
    np.arange(12).reshape((3, 4)),
    index=["Ohio", "Colorado", "New York"],
    columns=["one", "two", "three", "four"]
)

data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


In [18]:
def transform(x):
    return x[:4].upper()

data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

In [19]:
# Assign back to modify in place
data.index = data.index.map(transform)

data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


In [20]:
# rename: new object, original unchanged
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


In [21]:
# rename with dicts: rename only specific labels
data.rename(
    index={"OHIO": "INDIANA"},
    columns={"three": "peekaboo"}
)

,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11
